## Modelo LightGBM
LightGBM es una técnica de _gradient boosting_ desarrollada por Microsoft que utiliza algoritmos de aprendizaje basados ​​en árboles de decisión. Está diseñado para ofrecer las siguientes ventajas:
- Mayor velocidad de entrenamiento y mayor eficiencia.
- Menor consumo de memoria.
- Mayor precisión.
- Compatibilidad con aprendizaje en paralelo, distribuido y mediante GPU.
- Capacidad para manejar datos a gran escala.
Su [documentación](https://lightgbm.readthedocs.io/en/stable/) y [código fuente](https://github.com/lightgbm-org/LightGBM) son públicos, por lo que se pueden consultar ambas fuentes para más información.

Para lograr mejores predicciones, en esta notebook vamos a desarrollar y comparar dos modelos:

1. Un modelo excluyendo por completo las variables de la zona de strikes.
2. Un modelo implementando *feature engineering* para corregir espacialmente dichas variables.

### Carga de librerías

In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path
from flaml.automl.automl import AutoML

ROOT = Path('..')

## Primer modelo

### Definición de variables predictoras y respuesta

In [ ]:
temp1 = pl.read_parquet(ROOT/"data"/"processed"/"temporada1_limpio.parquet")

columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing']
columnas_contaminadas = [
    'sz_top', 'sz_bot', 'sz_mid', 'strike_zone_size', 'relative_height', 'pitch_location',
    'is_strike_zone', 'is_shadow_zone', 'distance_to_corner'
]
todas_excluidas = columnas_excluir + columnas_contaminadas

feature_cols = [c for c in temp1.columns if c not in todas_excluidas]
print(f"Features usadas ({len(feature_cols)}):", feature_cols)

### Entrenamiento con FLAML

In [ ]:
X = temp1.select(feature_cols).to_pandas()
y = temp1.select('swing').to_pandas()['swing']

automl = AutoML()

automl_settings = {
    "time_budget": 600,
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["lgbm"],
    "eval_method": "cv",
    "n_splits": 5,
    "early_stop": True,
    "seed": 42,
}

automl.fit(X_train=X, y_train=y, **automl_settings)

mejor_lgbm = automl.model.estimator

print("Mejor modelo:", mejor_lgbm)
print("Mejores hiperparámetros:", automl.best_config)
print("Mejor logloss (CV):", automl.best_loss)

### Importancia de variables

In [ ]:
importancias = mejor_lgbm.feature_importances_

df_importancias = pl.DataFrame(
    {"variable": X.columns, "importancia": importancias}
)

df_importancias = df_importancias.sort(by="importancia", descending=True)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10))

Según el primer modelo, `movement_complexity` que es una medida de cuánto se mueve la bola lanzada por el pitcher es, junto a `plate_z` (posición vertical de la pelota cuando cruza el _home plate_) son las variables más importantes para predecir la probabilidad de swing.

### Predicción para Kaggle

In [ ]:
temporada2 = pl.read_parquet(ROOT/"data"/"processed"/"temporada2_limpio.parquet").to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2['pitch_id']

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.drop(columns=[col for col in todas_excluidas if col in temporada2.columns])

# Alineamos las variables categóricas con las mismas categorías que en entrenamiento
columnas_texto = X.select_dtypes(include=['category']).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(X_test[col], categories=X[col].cat.categories)

# Reordenamos las columnas para que coincidan EXACTAMENTE con las de entrenamiento
X_test = X_test[mejor_lgbm.feature_names_in_]

# Predecimos las probabilidades
pred_test = mejor_lgbm.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing_probability': pred_test
})

Guardamos el archivo de predicciones.

In [ ]:
ruta_prediccion = ROOT / "outputs" / "lightgbm_1" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

## Segundo intento — variables de zona de strike promediadas por bateador

In [ ]:
temp1_promedio = pl.read_parquet(ROOT/"data"/"processed"/"temporada1_limpio_promedio.parquet")

### Definición de variables predictoras y respuesta

`temporada1_limpio_promedio.parquet` ya viene con las variables espaciales curadas (reemplazando a las contaminadas), por eso acá no repetimos `columnas_contaminadas`. Sumamos `balls`/`strikes` a la exclusión porque quedaron combinadas en la variable `count` (string). Las variables categóricas ya vienen tipadas como tal desde el preprocesamiento en Polars.

In [ ]:
columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing', 'sz_top_mean', 'sz_bot_mean', 'balls', 'strikes']

feature_cols = [c for c in temp1_promedio.columns if c not in columnas_excluir]
print(f"Features usadas ({len(feature_cols)}):", feature_cols)

### Entrenamiento con FLAML

In [ ]:
X = temp1_promedio.select(feature_cols).to_pandas()
y = temp1_promedio.select('swing').to_pandas()['swing']

automl = AutoML()

automl_settings = {
    "time_budget": 600,
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["lgbm"],
    "eval_method": "cv",
    "n_splits": 5,
    "early_stop": True,
    "seed": 42,
}

automl.fit(X_train=X, y_train=y, **automl_settings)

mejor_lgbm2 = automl.model.estimator

print("Mejor modelo:", mejor_lgbm2)
print("Mejores hiperparámetros:", automl.best_config)
print("Mejor logloss (CV):", automl.best_loss)

### Importancia de variables

In [ ]:
importancias = mejor_lgbm2.feature_importances_

df_importancias = pl.DataFrame(
    {"variable": X.columns, "importancia": importancias}
)

df_importancias = df_importancias.sort(by="importancia", descending=True)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10))

### Predicción para Kaggle

In [ ]:
temporada2 = pl.read_parquet(ROOT/"data"/"processed"/"temporada2_limpio_promedio.parquet")

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2['pitch_id']

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.select(feature_cols).to_pandas()

# Alineamos las variables categóricas con las mismas categorías que en entrenamiento
columnas_texto = X.select_dtypes(include=['category']).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(X_test[col], categories=X[col].cat.categories)

# Reordenamos las columnas para que coincidan EXACTAMENTE con las de entrenamiento
X_test = X_test[mejor_lgbm2.feature_names_in_]

# Predecimos las probabilidades
pred_test = mejor_lgbm2.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing_probability': pred_test
})

Guardamos el archivo de predicciones.

In [ ]:
ruta_prediccion = ROOT / "outputs" / "lightgbm_2" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)